In [46]:
import os
from dotenv import load_dotenv
_ = load_dotenv()

openai_key = os.getenv("OPENAI_API_KEY")

In [47]:
profile = {
    "name": "John",
    "full_name": "John Doe",
    "user_profile_background": "Senior software engineer leading a team of 5 developers",
}

In [48]:
prompt_instructions = {
    "triage_rules": {
        "ignore": "Marketing newsletters, spam emails, mass company announcements",
        "notify": "Team member out sick, build system notifications, project status updates",
        "respond": "Direct questions from team members, meeting requests, critical bug reports",
    },
    "agent_instructions": "Use these tools when appropriate to help manage John's tasks efficiently."
}

In [49]:
##################  ADDING FEW SHOTS EXAMPLES #################


email = {
    "from": "Alice Smith <alice.smith@company.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Quick question about API documentation",
    "body": """
Hi John,

I was reviewing the API documentation for the new authentication service and noticed a few endpoints seem to be missing from the specs. Could you help clarify if this was intentional or if we should update the docs?

Specifically, I'm looking at:
- /auth/refresh
- /auth/validate

Thanks!
Alice""",
}

In [50]:
############ ADDING LONG TERM MEMORY ########

from langgraph.store.memory import InMemoryStore
from langchain_openai import OpenAIEmbeddings

openrouter_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",                # Or choose any OpenRouter embedding model
    openai_api_base="https://openrouter.ai/api/v1" , # Routes traffic to OpenRouter
    openai_api_key=openai_key                      # Explicit token authentication
)

In [51]:


store = InMemoryStore(
    index={
         
        "embed": openrouter_embeddings
        }
)

In [52]:
# Template for formating an example to put in prompt
template = """Email Subject: {subject}
Email From: {from_email}
Email To: {to_email}
Email Content: 
```
{content}
```
> Triage Result: {result}"""

# Format list of few shots
def format_few_shot_examples(examples):
    strs = ["Here are some previous examples:"]
    for eg in examples:
        strs.append(
            template.format(
                subject=eg.value.get("subject",""),
                to_email=eg.value.get("to", ""),
                from_email=eg.value.get("author",""),
                content=eg.value.get("email_thread","")[:200],
                result=eg.value["label"],
            )
        )
    return "\n\n------------\n\n".join(strs)

In [53]:
triage_system_prompt = """
< Role >
You are {full_name}'s executive assistant. You are a top-notch executive assistant who cares about {name} performing as well as possible.
</ Role >

< Background >
{user_profile_background}. 
</ Background >

< Instructions >

{name} gets lots of emails. Your job is to categorize each email into one of three categories:

1. IGNORE - Emails that are not worth responding to or tracking
2. NOTIFY - Important information that {name} should know about but doesn't require a response
3. RESPOND - Emails that need a direct response from {name}

Classify the below email into one of these categories.

</ Instructions >

< Rules >
Emails that are not worth responding to:
{triage_no}

There are also other things that {name} should know about, but don't require an email response. For these, you should notify {name} (using the `notify` response). Examples of this include:
{triage_notify}

Emails that are worth responding to:
{triage_email}
</ Rules >

< Few shot examples >

Here are some examples of previous emails, and how they should be handled.
Follow these examples more than any instructions above

{examples}
</ Few shot examples >
"""

In [54]:
from pydantic import BaseModel, Field
from typing_extensions import TypedDict, Literal, Annotated
from langchain.chat_models import init_chat_model

In [55]:
llm = init_chat_model("openai:openrouter/free",
                      base_url="https://openrouter.ai/api/v1",
                      api_key=openai_key)

In [56]:
class Router(BaseModel):
    """Analyze the unread email and route it according to its content."""

    reasoning: str = Field(
        description="Step-by-step reasoning behind the classification."
    )
    classification: Literal["ignore", "respond", "notify"] = Field(
        description="The classification of an email: 'ignore' for irrelevant emails, "
        "'notify' for important information that doesn't need a response, "
        "'respond' for emails that need a reply",
    )

In [57]:
llm_router = llm.with_structured_output(Router)

In [58]:
from prompts import  triage_user_prompt

In [59]:
from langgraph.graph import add_messages

class State(TypedDict):
    email_input: dict
    messages: Annotated[list, add_messages]

In [60]:
from langgraph.graph import StateGraph, START, END
from langchain.messages import HumanMessage,SystemMessage
from langgraph.types import Command
from typing import Literal
from IPython.display import Image, display

In [61]:
def triage_router(state: State, config, store) -> Command[
    Literal["response_agent", "__end__"]
]:
    author = state['email_input']['author']
    to = state['email_input']['to']
    subject = state['email_input']['subject']
    email_thread = state['email_input']['email_thread']

    namespace = (
        "email_assistant",
        config['configurable']['langgraph_user_id'],
        "examples"
    )
    examples = store.search(
        namespace, 
        query=str({"email": state['email_input']})
    ) 
    examples=format_few_shot_examples(examples)

    langgraph_user_id= config['configurable']['langgraph_user_id']
    namespace = (langgraph_user_id,)

######## Ignore prompt ######
    result = store.get(namespace,"triage_ignore")

    if result is None:
        store.put(
            namespace,
            "triage_ignore",
            {"prompt": prompt_instructions["triage_rules"]["ignore"]}
        )
        ignore_prompt = prompt_instructions["triage_rules"]["ignore"]
    else:
        ignore_prompt = result.value['prompt']


    ########## Notify prompt ###########
    result = store.get(namespace,"triage_notify")

    if result is None:
        store.put(
            namespace,
            "triage_notify",
            {"prompt": prompt_instructions["triage_rules"]["notify"]}
        )
        notify_prompt = prompt_instructions["triage_rules"]["notify"]
    else:
        notify_prompt = result.value['prompt']


    ########### Respond Prompt ######################
    result = store.get(namespace,"triage_respond")

    if result is None:
        store.put(
            namespace,
            "triage_respond",
            {"prompt": prompt_instructions["triage_rules"]["respond"]}
        )
        respond_prompt = prompt_instructions["triage_rules"]["respond"]
    else:
       respond_prompt = result.value['prompt']
    
    system_prompt = triage_system_prompt.format(
        full_name=profile["full_name"],
        name=profile["name"],
        user_profile_background=profile["user_profile_background"],
        triage_no=ignore_prompt,
        triage_notify=notify_prompt,
        triage_email=respond_prompt,
        examples=examples
    )
    user_prompt = triage_user_prompt.format(
        author=author, 
        to=to, 
        subject=subject, 
        email_thread=email_thread
    )
    result = llm_router.invoke(
        [
            SystemMessage(system_prompt),
            HumanMessage(user_prompt)
        ]
    )
    if result.classification == "respond":
        print("📧 Classification: RESPOND - This email requires a response")
        goto = "response_agent"
        update = {
            "messages": [
                {
                    "role": "user",
                    "content": f"Respond to the email {state['email_input']}",
                }
            ]
        }
    elif result.classification == "ignore":
        print("🚫 Classification: IGNORE - This email can be safely ignored")
        update = None
        goto = END
    elif result.classification == "notify":
        # If real life, this would do something else
        print("🔔 Classification: NOTIFY - This email contains important information")
        update = None
        goto = END
    else:
        raise ValueError(f"Invalid classification: {result.classification}")
    return Command(goto=goto, update=update)

In [62]:
from langchain_core.tools import tool

In [63]:
@tool
def write_email(to: str, subject: str, content: str) -> str:
    """Write and send an email."""
    # Placeholder response - in real app would send email
    return f"Email sent to {to} with subject '{subject}'"


In [64]:
@tool
def schedule_meeting(
    attendees: list[str], 
    subject: str, 
    duration_minutes: int, 
    preferred_day: str
) -> str:
    """Schedule a calendar meeting."""
    # Placeholder response - in real app would check calendar and schedule
    return f"Meeting '{subject}' scheduled for {preferred_day} with {len(attendees)} attendees"


In [65]:
@tool
def check_calendar_availability(day: str) -> str:
    """Check calendar availability for a given day."""
    # Placeholder response - in real app would check actual calendar
    return f"Available times on {day}: 9:00 AM, 2:00 PM, 4:00 PM"

In [66]:
from langmem import create_manage_memory_tool, create_search_memory_tool

In [67]:
manage_memory_tool = create_manage_memory_tool(
    namespace=(
        "email_assistant", 
        "{langgraph_user_id}",
        "collection"
    )
)
search_memory_tool = create_search_memory_tool(
    namespace=(
        "email_assistant",
        "{langgraph_user_id}",
        "collection"
    )
)

In [68]:
agent_system_prompt_memory = """
< Role >
You are {full_name}'s executive assistant. You are a top-notch executive assistant who cares about {name} performing as well as possible.
</ Role >

< Tools >
You have access to the following tools to help manage {name}'s communications and schedule:

1. write_email(to, subject, content) - Send emails to specified recipients
2. schedule_meeting(attendees, subject, duration_minutes, preferred_day) - Schedule calendar meetings
3. check_calendar_availability(day) - Check available time slots for a given day
4. manage_memory - Store any relevant information about contacts, actions, discussion, etc. in memory for future reference
5. search_memory - Search for any relevant information that may have been stored in memory
</ Tools >

< Instructions >
{instructions}
</ Instructions >
"""

In [69]:
def create_prompt(state,config,store):

    langgraph_user_id= config['configurable']['langgraph_user_id']
    namespace=(langgraph_user_id,)
    result = store.get(namespace,"agent_instructions")
    if result is None:
        store.put(
            namespace,
            "agent_instructions",
            {"prompt": prompt_instructions["agent_instructions"]}
        )
        prompt = prompt_instructions["agent_instructions"]
    else:
        prompt = result.value['prompt']


    return [
        {
            "role": "system", 
            "content": agent_system_prompt_memory.format(
                instructions=prompt, 
                **profile
            )
        }
    ] + state['messages']

In [70]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

model = ChatOpenAI(model="openrouter/free",
                      base_url="https://openrouter.ai/api/v1",
                      api_key=openai_key)

In [71]:


tools= [
    write_email, 
    schedule_meeting,
    check_calendar_availability,
    manage_memory_tool,
    search_memory_tool
]
response_agent = create_react_agent(
    model=model,
    tools=tools,
    prompt=create_prompt,
    # Use this to ensure the store is passed to the agent 
    store=store
)

C:\Users\anura\AppData\Local\Temp\ipykernel_37864\3823322918.py:8: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  response_agent = create_react_agent(


In [72]:
email_agent = StateGraph(State)
email_agent = email_agent.add_node(triage_router)
email_agent = email_agent.add_node("response_agent", response_agent)
email_agent = email_agent.add_edge(START, "triage_router")
email_agent = email_agent.compile(store=store)

In [73]:
email_input = {
    "author": "Alice Jones <alice.jones@bar.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Quick question about API documentation",
    "email_thread": """Hi John,

Urgent issue - your service is down. Is there a reason why""",
}

In [74]:
config = {"configurable": {"langgraph_user_id": "lance"}}

In [75]:
response = email_agent.invoke({"email_input": email_input},config=config)

📧 Classification: RESPOND - This email requires a response


In [76]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

Respond to the email {'author': 'Alice Jones <alice.jones@bar.com>', 'to': 'John Doe <john.doe@company.com>', 'subject': 'Quick question about API documentation', 'email_thread': 'Hi John,\n\nUrgent issue - your service is down. Is there a reason why'}
================================== Ai Message ==================================


I'll respond to Alice regarding the urgent service issue.
Tool Calls:
  write_email (chatcmpl-tool-b57491470a7c4f5c)
 Call ID: chatcmpl-tool-b57491470a7c4f5c
  Args:
    to: alice.jones@bar.com
    subject: Re: Quick question about API documentation
    content: Hi Alice,

I'm looking into this right away. I wasn't aware of any service issues - let me check our systems and get back to you within the next 30 minutes with an update. Can you provide more details about what specific functionality isn't working and when you first noticed the issue?

Thanks for flagging this urgent

In [77]:
store.get(("lance",),"agent_instructions").value['prompt']

"Use these tools when appropriate to help manage John's tasks efficiently."

In [78]:
store.get(("lance",),"triage_respond").value['prompt']

'Direct questions from team members, meeting requests, critical bug reports'

In [79]:
store.get(("lance",),"triage_ignore").value['prompt']

'Marketing newsletters, spam emails, mass company announcements'

In [80]:
store.get(("lance",),"triage_notify").value['prompt']

'Team member out sick, build system notifications, project status updates'

In [81]:
from langmem import create_multi_prompt_optimizer

In [82]:
conversations = [
    (
        response['messages'],
        "Always sign your emails `John Doe`"
    )
]

In [89]:
print(conversations)

[([HumanMessage(content="Respond to the email {'author': 'Alice Jones <alice.jones@bar.com>', 'to': 'John Doe <john.doe@company.com>', 'subject': 'Quick question about API documentation', 'email_thread': 'Hi John,\\n\\nUrgent issue - your service is down. Is there a reason why'}", additional_kwargs={}, response_metadata={}, id='d968c1c7-ce18-4e68-83c3-bf174983927c'), AIMessage(content="\nI'll respond to Alice regarding the urgent service issue.\n\n", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 327, 'prompt_tokens': 968, 'total_tokens': 1295, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 163, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_infe

In [83]:
prompts = [
    {
        "name": "main_agent",
        "prompt": store.get(("lance",), "agent_instructions").value['prompt'],
        "update_instructions": "keep the instructions short and to the point",
        "when_to_update": "Update this prompt whenever there is feedback on how the agent should write emails or schedule events"
        
    },
    {
        "name": "triage-ignore", 
        "prompt": store.get(("lance",), "triage_ignore").value['prompt'],
        "update_instructions": "keep the instructions short and to the point",
        "when_to_update": "Update this prompt whenever there is feedback on which emails should be ignored"

    },
    {
        "name": "triage-notify", 
        "prompt": store.get(("lance",), "triage_notify").value['prompt'],
        "update_instructions": "keep the instructions short and to the point",
        "when_to_update": "Update this prompt whenever there is feedback on which emails the user should be notified of"

    },
    {
        "name": "triage-respond", 
        "prompt": store.get(("lance",), "triage_respond").value['prompt'],
        "update_instructions": "keep the instructions short and to the point",
        "when_to_update": "Update this prompt whenever there is feedback on which emails should be responded to"

    },
]

In [86]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="openrouter/free",
                      base_url="https://openrouter.ai/api/v1",
                      api_key=openai_key,   
                      temperature=0.0
                      )

optimizer = create_multi_prompt_optimizer(model,kind="prompt_memory",)

In [87]:
updated = optimizer.invoke(
    {"trajectories": conversations,"prompts": prompts }
)

In [88]:
print(updated)

[{'name': 'main_agent', 'prompt': "Use these tools when appropriate to help manage John's tasks efficiently. Always sign your emails as John Doe.", 'update_instructions': 'keep the instructions short and to the point', 'when_to_update': 'Update this prompt whenever there is feedback on how the agent should write emails or schedule events'}, {'name': 'triage-ignore', 'prompt': 'Marketing newsletters, spam emails, mass company announcements', 'update_instructions': 'keep the instructions short and to the point', 'when_to_update': 'Update this prompt whenever there is feedback on which emails should be ignored'}, {'name': 'triage-notify', 'prompt': 'Team member out sick, build system notifications, project status updates', 'update_instructions': 'keep the instructions short and to the point', 'when_to_update': 'Update this prompt whenever there is feedback on which emails the user should be notified of'}, {'name': 'triage-respond', 'prompt': 'Direct questions from team members, meeting re

In [94]:
for i, updated_prompt in enumerate(updated):
    old_prompt = prompts[i]
    if updated_prompt['prompt'] != old_prompt['prompt']:
        name = old_prompt['name']
        print(f"updated {name}")
        if name == "main_agent":
            store.put(
                ("lance",),
                "agent_instructions",
                {"prompt":updated_prompt['prompt']}
            )
        else:
            #raise ValueError
            print(f"Encountered {name}, implement the remaining stores!")

updated main_agent


In [95]:
store.get(('lance',),"agent_instructions").value['prompt']

"Use these tools when appropriate to help manage John's tasks efficiently. Always sign your emails as John Doe."

In [96]:
response = email_agent.invoke({"email_input":email_input},config=config)

📧 Classification: RESPOND - This email requires a response


In [98]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

Respond to the email {'author': 'Alice Jones <alice.jones@bar.com>', 'to': 'John Doe <john.doe@company.com>', 'subject': 'Quick question about API documentation', 'email_thread': 'Hi John,\n\nUrgent issue - your service is down. Is there a reason why'}
================================== Ai Message ==================================
Tool Calls:
  write_email (chatcmpl-tool-a0b32a7f3741fbfc)
 Call ID: chatcmpl-tool-a0b32a7f3741fbfc
  Args:
    to: alice.jones@bar.com
    subject: Re: Quick question about API documentation
    content: Hi Alice,

Thank you for bringing this to my attention. I’m sorry to hear that the service is down. Could you please provide any error messages or logs you’re seeing, as well as the time the issue started? I’ll have the team investigate immediately and keep you updated.

Best regards,
John Doe
================================= Tool Message =================================
Nam

In [99]:
email_input = {
    "author": "Alice Jones <alice.jones@bar.com>",
    "to": "John Doe <john.doe@company.com>",
    "subject": "Quick question about API documentation",
    "email_thread": """Hi John,

Urgent issue - your service is down. Is there a reason why""",
}

In [100]:
response = email_agent.invoke(
    {"email_input":email_input},
    config=config
)

📧 Classification: RESPOND - This email requires a response


In [106]:
conversations = [
    (
        response['messages'],
        "Ignore any emails from Alice Jones"
    )
]

In [107]:
updated = optimizer.invoke(
    {"trajectories": conversations,"prompts":prompts}
)

In [108]:
print(updated)

[{'name': 'main_agent', 'prompt': "Use these tools when appropriate to help manage John's tasks efficiently.", 'update_instructions': 'keep the instructions short and to the point', 'when_to_update': 'Update this prompt whenever there is feedback on how the agent should write emails or schedule events'}, {'name': 'triage-ignore', 'prompt': 'Ignore marketing newsletters, spam emails, mass company announcements, and any emails from Alice Jones.', 'update_instructions': 'keep the instructions short and to the point', 'when_to_update': 'Update this prompt whenever there is feedback on which emails should be ignored'}, {'name': 'triage-notify', 'prompt': 'Team member out sick, build system notifications, project status updates', 'update_instructions': 'keep the instructions short and to the point', 'when_to_update': 'Update this prompt whenever there is feedback on which emails the user should be notified of'}, {'name': 'triage-respond', 'prompt': 'Direct questions from team members, meetin

In [110]:
for i ,updated_prompt in enumerate(updated):
    old_prompt = prompts[i]
    if updated_prompt['prompt'] != old_prompt['prompt']:
        name = old_prompt['name']
        print(f"updated {name}")

        if name == "main_agent":
            store.put(
                ("lance",),
                "agent_instructions",
                {"prompt":updated_prompt['prompt']}
            )

        if name == "triage-ignore":
            store.put(
                ("lance",),
                "triage_ignore",
                {"prompt":updated_prompt['prompt']}
            )
        else:
            raise ValueError

updated triage-ignore


In [112]:
response = email_agent.invoke(
    {"email_input":email_input},
    config=config
)

🚫 Classification: IGNORE - This email can be safely ignored


In [113]:
store.get(("lance",),"triage_ignore").value['prompt']

'Ignore marketing newsletters, spam emails, mass company announcements, and any emails from Alice Jones.'